# 1. Training
Hyperparameter search, the proposed model on the five splits, the same network with other pooling functions, and the best configuration of the search on the five splits.

Run the cells from top to bottom. A run whose result file already exists is skipped, so the notebook can be stopped and started again.

In [1]:
import torch

from constants import *
from utils import get_device
from experiments import run_experiment
from hparam_search import run_hyperparameter_search
from disk_utils import load_results
from result_tables import per_seed_table, summary_table

In [2]:
USE_GPU = True
device = get_device(USE_GPU)
print(device)

# QUICK_TEST = True makes a tiny dry run of every step (1 epoch, 3 batches)
# and writes to results_quick_test/. Use it to check the setup.
QUICK_TEST = False

if QUICK_TEST:
    results_dir = QUICK_TEST_RESULTS_DIR
    num_epochs = 1
    max_train_batches = 3
else:
    results_dir = RESULTS_DIR
    num_epochs = NUM_EPOCHS
    max_train_batches = None

if QUICK_TEST:
    num_search_trials = 2
else:
    num_search_trials = NUM_SEARCH_TRIALS

cuda


## Hyperparameter search
Random search on split 0. Trial 0 is the configuration that was fixed beforehand. The configurations are compared by validation loss; the test subset is not used.

In [3]:
search_table = run_hyperparameter_search(
    device,
    num_trials=num_search_trials,
    seed=0,
    num_epochs=num_epochs,
    max_train_batches=max_train_batches,
    results_dir=results_dir
)

[hp_00 s0] already done, skipping
[hp_01 s0] already done, skipping
[hp_02 s0] already done, skipping
[hp_03 s0] already done, skipping
[hp_04 s0] already done, skipping
[hp_05 s0] already done, skipping
[hp_06 s0] already done, skipping
[hp_07 s0] already done, skipping
[hp_08 s0] already done, skipping
[hp_09 s0] already done, skipping
[hp_10 s0] already done, skipping
[hp_11 s0] already done, skipping
[hp_12 s0] already done, skipping
[hp_13 s0] already done, skipping
[hp_14 s0] already done, skipping
[hp_15 s0] already done, skipping
[hp_16 s0] already done, skipping
[hp_17 s0] already done, skipping
[hp_18 s0] already done, skipping
[hp_19 s0] already done, skipping


In [4]:
search_table

,lstm_hidden,emb_dim,learning_rate,dropout,batch_size,trial,best_epoch,val_loss,val_acc,n_params,seconds
16,64,32,0.0010,0.3,8,16,12,0.154977,0.948276,126892,111
19,128,32,0.0010,0.2,8,19,14,0.172298,0.948276,333676,121
18,256,32,0.0010,0.2,16,18,14,0.172786,0.931034,993004,68
4,256,32,0.0010,0.2,8,4,6,0.179699,0.936782,993004,73
6,128,64,0.0005,0.2,8,6,12,0.180111,0.936782,499532,111
7,64,64,0.0010,0.2,8,7,8,0.180492,0.948276,210828,89
1,64,32,0.0010,0.3,16,1,11,0.180583,0.931034,126892,62
0,128,32,0.0010,0.3,8,0,12,0.183518,0.936782,333676,110
8,128,32,0.0005,0.3,16,8,16,0.184491,0.936782,333676,77
11,256,64,0.0005,0.5,16,11,9,0.194752,0.925287,1322700,52


## Proposed model
BiLSTM with attention on all five note attributes (pitch class, accidental, octave, duration, measure position).

In [5]:
for seed in SEEDS:
    run_experiment(
        tag="full",
        seed=seed,
        device=device,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[full s0] already done, skipping
[full s1] already done, skipping
[full s2] already done, skipping
[full s3] already done, skipping
[full s4] already done, skipping


## Without packing
The LSTM also runs over the padding, as in the preliminary version.

In [6]:
for seed in SEEDS:
    run_experiment(
        tag="full_unpacked",
        seed=seed,
        device=device,
        packed=False,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[full_unpacked s0] already done, skipping
[full_unpacked s1] already done, skipping
[full_unpacked s2] already done, skipping
[full_unpacked s3] already done, skipping
[full_unpacked s4] already done, skipping


## Other pooling functions in place of attention

In [7]:
for seed in SEEDS:
    run_experiment(
        tag="mean_pool",
        seed=seed,
        device=device,
        pooling="mean",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[mean_pool s0] already done, skipping
[mean_pool s1] already done, skipping
[mean_pool s2] already done, skipping
[mean_pool s3] already done, skipping
[mean_pool s4] already done, skipping


In [8]:
for seed in SEEDS:
    run_experiment(
        tag="max_pool",
        seed=seed,
        device=device,
        pooling="max",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[max_pool s0] already done, skipping
[max_pool s1] already done, skipping
[max_pool s2] already done, skipping
[max_pool s3] already done, skipping
[max_pool s4] already done, skipping


In [9]:
for seed in SEEDS:
    run_experiment(
        tag="last_pool",
        seed=seed,
        device=device,
        pooling="last",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[last_pool s0] already done, skipping
[last_pool s1] already done, skipping
[last_pool s2] already done, skipping
[last_pool s3] already done, skipping
[last_pool s4] already done, skipping


## Configuration with the lowest validation loss in the search
The proposed model keeps the configuration that was fixed beforehand. As a check, the configuration that came first in the hyperparameter search is also trained on the five splits. If the two give the same accuracy, the choice of the configuration does not matter.

In [10]:
# the search table is sorted by validation loss: the first row is the best trial
best_trial = search_table.iloc[0]
print(best_trial)

for seed in SEEDS:
    run_experiment(
        tag="search_best",
        seed=seed,
        device=device,
        lstm_hidden=int(best_trial["lstm_hidden"]),
        emb_dim=int(best_trial["emb_dim"]),
        learning_rate=float(best_trial["learning_rate"]),
        dropout=float(best_trial["dropout"]),
        batch_size=int(best_trial["batch_size"]),
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

lstm_hidden          64.000000
emb_dim              32.000000
learning_rate         0.001000
dropout               0.300000
batch_size            8.000000
trial                16.000000
best_epoch           12.000000
val_loss              0.154977
val_acc               0.948276
n_params         126892.000000
seconds             111.000000
Name: 16, dtype: float64
[search_best s0] Epoch 01 | Train Loss: 2.0464, Train Acc: 0.3604 | Val Loss: 1.2019, Val Acc: 0.6724 | LR: 1.0e-03
[search_best s0] Epoch 02 | Train Loss: 0.8287, Train Acc: 0.7601 | Val Loss: 0.4845, Val Acc: 0.8506 | LR: 1.0e-03
[search_best s0] Epoch 03 | Train Loss: 0.4907, Train Acc: 0.8635 | Val Loss: 0.4318, Val Acc: 0.8333 | LR: 1.0e-03
[search_best s0] Epoch 04 | Train Loss: 0.3980, Train Acc: 0.8635 | Val Loss: 0.3455, Val Acc: 0.8851 | LR: 1.0e-03
[search_best s0] Epoch 05 | Train Loss: 0.3477, Train Acc: 0.8893 | Val Loss: 0.2742, Val Acc: 0.8851 | LR: 1.0e-03
[search_best s0] Epoch 06 | Train Loss: 0.2800, Train 

## Results so far
Mean accuracy and macro-F1 over the five splits, with the half-width of the 95% confidence interval.

In [11]:
results = load_results(results_dir)
per_seed = per_seed_table(results)
summary = summary_table(per_seed)
summary[["tag", "perturbation", "n", "acc_mean", "acc_ci", "f1_mean", "f1_ci"]]

,tag,perturbation,n,acc_mean,acc_ci,f1_mean,f1_ci
0,bigram__logreg,,5,0.900571,0.008090,0.896318,0.010570
1,bigram__svm_rbf,,5,0.915429,0.019038,0.912351,0.022586
2,full,,5,0.920000,0.029254,0.914877,0.031895
3,full,shuffle,5,0.801143,0.023745,0.796994,0.025061
4,full,transpose+13,5,0.169143,0.049311,0.080260,0.033280
...,...,...,...,...,...,...,...
57,train_shuffle,,5,0.891429,0.019431,0.886884,0.026632
58,train_shuffle,shuffle,5,0.893714,0.018502,0.891226,0.023405
59,transpose_aug,,5,0.540571,0.238118,0.530828,0.249447
60,transpose_aug,transpose+4,5,0.531429,0.207891,0.521824,0.215062
